# BirdCLEF+ 2026 - Baseline Training (v2: + Soundscape Data)

Train an EfficientNet-B0 baseline on mel spectrograms.
**Now includes soundscape training data** to reduce the domain gap between clean recordings and real-world test soundscapes.

**Environment:** Kaggle GPU notebook (T4/P100)

In [ ]:
import os
import sys

# Detect environment
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    sys.path.insert(0, "/kaggle/input/datasets/stochastix/birdclef-src")
    DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
    OUTPUT_DIR = "/kaggle/working"
else:
    sys.path.insert(0, "..")
    DATA_DIR = "../data"
    OUTPUT_DIR = "../models"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from torch.amp import GradScaler, autocast
from sklearn.model_selection import train_test_split
import yaml
from tqdm.auto import tqdm

from src.dataset import BirdCLEFDataset, SoundscapeDataset, load_taxonomy
from src.model import get_model
from src.transforms import get_audio_transforms, get_mixup_fn
from src.evaluate import evaluate_roc_auc
from src.utils import set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

In [ ]:
# Load config
config_path = "/kaggle/input/datasets/stochastix/birdclef-src/config/default.yaml" if ON_KAGGLE else "../config/default.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

# Override data paths for current environment
config["data"]["train_audio_dir"] = os.path.join(DATA_DIR, "train_audio")
config["data"]["train_csv"] = os.path.join(DATA_DIR, "train.csv")
config["data"]["taxonomy_csv"] = os.path.join(DATA_DIR, "taxonomy.csv")
config["data"]["train_soundscapes_dir"] = os.path.join(DATA_DIR, "train_soundscapes")
config["data"]["soundscape_labels_csv"] = os.path.join(DATA_DIR, "train_soundscapes_labels.csv")

# Optimized settings
EPOCHS = 18
BATCH_SIZE = 64  # doubled from 32 — fits in T4x2 (32GB)
LR = config["training"]["lr"]
EARLY_STOP_PATIENCE = 3  # stop if no improvement for 3 epochs

set_seed(config["data"]["seed"])
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LR}, Early stop: {EARLY_STOP_PATIENCE}")

## Load Data

In [ ]:
# Load species list
species_list = load_taxonomy(config["data"]["taxonomy_csv"])
print(f"Species: {len(species_list)}")

# Load metadata
train_df = pd.read_csv(config["data"]["train_csv"])
print(f"Total recordings: {len(train_df)}")

# Filter by rating
min_rating = config["data"].get("min_rating", 0)
if min_rating > 0 and "rating" in train_df.columns:
    train_df = train_df[train_df["rating"] >= min_rating].reset_index(drop=True)
    print(f"After rating filter: {len(train_df)}")

# Remove species with only 1 sample (can't stratify)
label_counts = train_df["primary_label"].value_counts()
rare_labels = label_counts[label_counts < 2].index
rare_df = train_df[train_df["primary_label"].isin(rare_labels)]
main_df = train_df[~train_df["primary_label"].isin(rare_labels)]
print(f"Rare species (1 sample, added to train): {len(rare_labels)}")

# Stratified split on the rest
main_train, val_split = train_test_split(
    main_df, test_size=0.2,
    stratify=main_df["primary_label"],
    random_state=config["data"]["seed"]
)

# Add rare species to training set
train_split = pd.concat([main_train, rare_df], ignore_index=True)
print(f"Train: {len(train_split)}, Val: {len(val_split)}")

## Create Datasets & DataLoaders

In [ ]:
audio_transforms = get_audio_transforms(config.get("augmentation", {}))

# Clean recordings dataset
train_dataset = BirdCLEFDataset(
    df=train_split,
    audio_dir=config["data"]["train_audio_dir"],
    config=config,
    species_list=species_list,
    audio_transforms=audio_transforms,
    is_train=True,
)
print(f"Train recordings: {len(train_dataset)}")

# Add soundscape data if available
soundscape_dir = config["data"]["train_soundscapes_dir"]
soundscape_csv = config["data"]["soundscape_labels_csv"]

if os.path.exists(soundscape_dir) and os.path.exists(soundscape_csv):
    soundscape_labels = pd.read_csv(soundscape_csv)
    print(f"Soundscape labels loaded: {len(soundscape_labels)} annotations")
    print(f"Unique soundscape files: {soundscape_labels['filename'].nunique()}")

    soundscape_dataset = SoundscapeDataset(
        soundscape_dir=soundscape_dir,
        config=config,
        species_list=species_list,
        labels_df=soundscape_labels,
        is_test=False,
        augment=True,
        audio_transforms=audio_transforms,
    )
    print(f"Soundscape windows: {len(soundscape_dataset)}")

    # Combine both datasets
    combined_train = ConcatDataset([train_dataset, soundscape_dataset])
    print(f"Combined training samples: {len(combined_train)}")
else:
    combined_train = train_dataset
    print("No soundscape data found — using clean recordings only")

# Validation stays clean recordings only
val_dataset = BirdCLEFDataset(
    df=val_split,
    audio_dir=config["data"]["train_audio_dir"],
    config=config,
    species_list=species_list,
    audio_transforms=None,
    is_train=False,
)

train_loader = DataLoader(
    combined_train, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=8, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=8, pin_memory=True
)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Verify a batch
spec, labels = next(iter(train_loader))
print(f"Spectrogram shape: {spec.shape}")
print(f"Labels shape: {labels.shape}")

## Model & Optimizer

In [ ]:
model = get_model(config["model"]).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR,
    weight_decay=config["training"].get("weight_decay", 0.01)
)

warmup_epochs = config["training"].get("warmup_epochs", 2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - warmup_epochs, eta_min=LR * 0.01
)

criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler("cuda") if device.type == "cuda" else None

# Mixup
mixup_fn = None
aug_cfg = config.get("augmentation", {})
if aug_cfg.get("mixup_alpha", 0) > 0:
    mixup_fn = get_mixup_fn(aug_cfg["mixup_alpha"])

## Training Loop

In [ ]:
best_auc = 0.0
history = {"loss": [], "val_auc": [], "lr": []}
no_improve_count = 0

for epoch in range(1, EPOCHS + 1):
    # Warmup LR
    if epoch <= warmup_epochs:
        warmup_lr = LR * epoch / warmup_epochs
        for pg in optimizer.param_groups:
            pg["lr"] = warmup_lr

    # Train
    model.train()
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for specs, labels in pbar:
        specs, labels = specs.to(device), labels.to(device)

        if mixup_fn is not None and np.random.random() < aug_cfg.get("mixup_prob", 0.5):
            specs, labels, _ = mixup_fn(specs, labels)

        optimizer.zero_grad()
        if scaler:
            with autocast("cuda"):
                logits = model(specs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(specs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({"loss": f"{running_loss/num_batches:.4f}"})

    avg_loss = running_loss / max(num_batches, 1)

    if epoch > warmup_epochs:
        scheduler.step()

    # Validate
    val_auc = evaluate_roc_auc(model, val_loader, device, species_list)

    current_lr = optimizer.param_groups[0]["lr"]
    history["loss"].append(avg_loss)
    history["val_auc"].append(val_auc)
    history["lr"].append(current_lr)

    print(f"Epoch {epoch}: loss={avg_loss:.4f}, val_auc={val_auc:.4f}, lr={current_lr:.6f}")

    if val_auc > best_auc:
        best_auc = val_auc
        no_improve_count = 0
        save_path = os.path.join(OUTPUT_DIR, "best_model.pth")
        torch.save(model.state_dict(), save_path)
        print(f"  -> Saved best model (AUC: {best_auc:.4f})")
    else:
        no_improve_count += 1
        if no_improve_count >= EARLY_STOP_PATIENCE and epoch > warmup_epochs:
            print(f"  -> Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs")
            break

print(f"\nBest Val ROC-AUC: {best_auc:.4f}")
print(f"Stopped at epoch {epoch}/{EPOCHS}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history["loss"])
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_auc"])
axes[1].set_title("Validation ROC-AUC")
axes[1].set_xlabel("Epoch")

axes[2].plot(history["lr"])
axes[2].set_title("Learning Rate")
axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

## Sanity Check

In [ ]:
import librosa.display

# Load best model
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pth"), map_location=device))
model.eval()

# Predict on a few validation samples
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

with torch.no_grad():
    for i in range(min(6, len(val_dataset))):
        spec, label = val_dataset[i]
        logits = model(spec.unsqueeze(0).to(device))
        probs = torch.sigmoid(logits).cpu().numpy()[0]

        # True labels
        true_species = [species_list[j] for j in np.where(label.numpy() > 0.5)[0]]
        # Top-3 predictions
        top3_idx = np.argsort(probs)[-3:][::-1]
        top3 = [(species_list[j], probs[j]) for j in top3_idx]

        # Plot spectrogram
        axes[i].imshow(spec[0].numpy(), aspect="auto", origin="lower", cmap="viridis")
        axes[i].set_title(
            f"True: {', '.join(true_species)}\n"
            f"Pred: {top3[0][0]}({top3[0][1]:.2f}), {top3[1][0]}({top3[1][1]:.2f})",
            fontsize=9
        )
        axes[i].axis("off")

plt.suptitle("Validation Predictions (Sanity Check)", fontsize=14)
plt.tight_layout()
plt.show()